This notebook contains the code for dataset exploration and preprocessing.

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import scipy.io

root = os.path.dirname(os.getcwd())

## Preparing the data for preprocessing

In [ ]:
#making the records and machine measurements fit our subset of data
#since the full machine measurements and records contain more files than necessary

subset_root = r"\data\mimic_subset" #adjust to path

#load CSVs first
record_list = pd.read_csv(os.path.join(subset_root, "full_record_list.csv"))
machine_measurements = pd.read_csv(os.path.join(subset_root, "full_machine_measurements.csv"))

print(f"{len(record_list)} records to check")

#check existence directly from the path column
record_list_subset = record_list[
    record_list["path"].apply(
        lambda p: os.path.exists(os.path.join(subset_root, p + ".hea"))
    )
]

print(f"{len(record_list_subset)} records on disk")

machine_measurements_subset = machine_measurements[
    machine_measurements["study_id"].isin(record_list_subset["study_id"])
]

print(f"machine_measurements: {len(machine_measurements_subset)} rows")

record_list_subset.to_csv(os.path.join(subset_root, "record_list_subset.csv"), index=False)
machine_measurements_subset.to_csv(os.path.join(subset_root, "machine_measurements_subset.csv"), index=False)

In [ ]:
#some files seem to miss either a .hea or .dat file, which makes the preprocessing crash
#this codes finds those files, which can then be manually removed

files_root = r"data\mimic_subset\files" #adjust to path

missing_dat = []
missing_hea = []

for dirpath, dirnames, filenames in os.walk(files_root):
    hea_files = {os.path.splitext(f)[0] for f in filenames if f.endswith(".hea")}
    dat_files = {os.path.splitext(f)[0] for f in filenames if f.endswith(".dat")}
    
    for name in hea_files - dat_files:
        missing_dat.append(os.path.join(dirpath, name))
    for name in dat_files - hea_files:
        missing_hea.append(os.path.join(dirpath, name))

print(f"Records missing .dat: {len(missing_dat)}")
print(f"Records missing .hea: {len(missing_hea)}")
for p in missing_dat[:10]:
    print(p)
for h in missing_hea[:20]:
    print(h)

In [ ]:
#to check the number of files in record list csv, for later comparison
df = pd.read_csv(r"data\mimic_subset\mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0\record_list.csv") #adjust to path
print(len(df))

In [ ]:
#to check the number of files in the subset, and they match!
files_root = r"mimic_subset\mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0\files" #adjust to path

hea_count = sum(
    1 for dirpath, dirnames, filenames in os.walk(files_root)
    for f in filenames if f.endswith(".hea")
)

print(f"Total .hea files: {hea_count}")

## Preprocessing (mainly done in terminal)

The next code was taken from the fairseq repository (https://github.com/Jwoo5/fairseq-signals/tree/master/scripts/preprocess/ecg) and ran in the terminal in order to preprocess the dataset into org, preprocessed, and segmented (splitting up the 10 second ECGs into 5 seconds), after which a split of approximately 80/10/10 was performed based on the subject ID. 

In [ ]:
# cd fairseq-signals\scripts\preprocess\ecg #adjust to path

# python mimic_iv_ecg_records.py `
#     --processed_root outputs\processed ` #adjust to path
#     --raw_root data\mimic_subset #adjust to path

In [ ]:
# python mimic_iv_ecg_signals.py `
#     --processed_root outputs\processed ` #adjust to path
#     --raw_root data\mimic_subset ` #adjust to path
#     --manifest_file outputs\manifest.csv ` #adjust to path
#     --no_parallel

In [ ]:
# cd fairseq-signals\scripts\preprocess #adjust to path

# python splits.py `
#     --strategy grouped `
#     --processed_root \ecg-thesis\outputs\processed ` #adjust to path
#     --group_col subject_id `
#     --filter_cols nan_any,constant_leads_any

In [ ]:
#checking if number of files still matches, and it does
files_root = r"\outputs\processed\preprocessed" #adjust to path

mat_count = sum(
    1 for dirpath, dirnames, filenames in os.walk(files_root)
    for f in filenames
    if f.endswith('.mat')
)

print(f"Total .mat files: {mat_count}")

In [ ]:
#checking number of files after segmentation
files_root = r"\outputs\processed\segmented" #adjust to path

seg_count = sum(
    1 for dirpath, dirnames, filenames in os.walk(files_root)
    for f in filenames
)

print(f"Total .seg files: {seg_count}")

In [ ]:
#checking structure of new preprocessed data

import pandas as pd
df = pd.read_csv(r"\outputs\processed\segmented_split.csv") #adjust to path
print(df.columns.tolist())
print(df.head(2))

In [ ]:
#checking the numbers for the split
segmented = pd.read_csv(r"outputs\processed\segmented_split.csv") #adjust to path
print(segmented['split'].value_counts())
print(segmented['split'].value_counts(normalize=True))

In [ ]:
#creating the manifest requried for the finetuning run. code taken from fairseq website again

#code used
#python manifests.py  
# --split_file_paths "outputs\processed\segmented_split.csv"  #adjust to path
# --save_dir "outputs\manifest" #adjust to path

In [ ]:
#checking if no data got lost (it didnt)
train = pd.read_csv(r"outputs\manifest\train.tsv", #adjust to path
                    sep='\t', skiprows=1, header=None)
valid = pd.read_csv(r"outputs\manifest\valid.tsv", #adjust to path
                    sep='\t', skiprows=1, header=None)
test = pd.read_csv(r"outputs\manifest\test.tsv", #adjust to path
                   sep='\t', skiprows=1, header=None)

print(f"train: {len(train)}")
print(f"test:  {len(test)}")
print(f"valid: {len(valid)}")
print(f"total: {len(train) + len(valid) + len(test)}")


In [ ]:
#checking the positive split of the manifest file per label

manifests_dir = r'\outputs\manifest' #adjust path

#checking how many samples are available for each label in the full dataset
y = np.load(os.path.join(manifests_dir, 'y.npy'))

counts = y.sum(axis=0)
for i, count in enumerate(counts):
    print(f"label {i}: {count}")

In [ ]:
#checking test set positive counts per label

manifests_dir = r'\outputs\manifest' #adjust path
seg_dir = r'\outputs\processed\segmented' #adjust path
label_def = r'\data\mimic_iv_ecg\labels\label_def.csv' #adjust path

y = np.load(os.path.join(manifests_dir, 'y.npy'))
label_cols = pd.read_csv(label_def)['name'].tolist()

with open(os.path.join(manifests_dir, 'test.tsv'), 'r') as f:
    lines = f.read().splitlines()

root = lines[0].strip()
fnames = [line.split('\t')[0] for line in lines[1:] if line.strip()]

#get idx for each test segment
test_indices = []
for fname in fnames:
    path = os.path.join(root, fname)
    mat = scipy.io.loadmat(path)
    test_indices.append(int(mat['idx'][0, 0]))

y_test = y[test_indices]
counts = y_test.sum(axis=0)

for i, (label, count) in enumerate(zip(label_cols, counts)):
    print(f"{i:2d}  {count:5d}  {label}")

In [ ]:
#creating the manifest file, y.npy, to use for training
#linking the .mat files back to their study diagnoses

seg_dir = r"\outputs\processed\segmented" #adjust path
manifests_dir = r'\outputs\manifest' #adjust path
label_dir = r'data\mimic_iv_ecg\labels\true_labels.csv' #adjust path

#load the labels
labels_df = pd.read_csv(label_dir).set_index('study_id')
label_cols = [c for c in labels_df.columns if c != 'idx']

#read every .mat file and record which study it belongs to
idx_to_study = {}
fnames = [f for f in os.listdir(seg_dir) if f.endswith('.mat')]
for i, fname in enumerate(fnames):
    mat = scipy.io.loadmat(os.path.join(seg_dir, fname))
    idx = int(mat['idx'][0, 0])
    study_id = int(re.search(r'_s(\d+)_', fname).group(1))
    idx_to_study[idx] = study_id
    if i % 1000 == 0:
        print(i, "out of", len(fnames), "files read")

#build y.npy: row i contains the labels for the study with idx i
max_idx = max(idx_to_study.keys())
y = np.zeros((max_idx + 1, len(label_cols)), dtype=int)
for idx, study_id in idx_to_study.items():
    y[idx] = labels_df.loc[study_id][label_cols].values.astype(int)

np.save(os.path.join(manifests_dir, 'y.npy'), y)
print("done, saved y.npy with shape", y.shape)